***Installing dependencies***

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes evaluate safetensors

# ***Imports, settings, and general parameters***



In [ ]:
import os
import math
import random
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, DataCollatorWithPadding, default_data_collator
from transformers import BitsAndBytesConfig
import evaluate

# PEFT
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

# پارامترهای عمومی
LLAMA_ID = "meta-llama/Llama-3.2-8B-Instruct"

EVAL_SAMPLE = 20   # برای zero/one-shot evaluation
TRAIN_SAMPLE_QLORA = 20  # برای آموزش QLoRA در نوت بوک؛ میتونی بزرگتر کنی اگر منابع داری

# device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# metric
acc_metric = evaluate.load("accuracy")


# ***Loading the MultiNLI (MNLI) dataset and working samples***

In [ ]:
# بارگذاری dataset (Hugging Face datasets اسمش multi_nli)
raw = load_dataset("multi_nli")

# نمونه‌های آموزشی/اعتبارسنجیِ کوچکی برای اجرا در Colab
train_full = raw["train"]
val_matched = raw["validation_matched"]


train10pct = train_full.shuffle(seed=42).select(range(int(len(train_full)*0.10)))

eval_subset = val_matched.shuffle(seed=42).select(range(min(EVAL_SAMPLE, len(val_matched))))

print("Train (10%) size:", len(train10pct))
print("Eval subset size:", len(eval_subset))


# ***Preparing the LLaMA tokenizer and prompt template creation function***

In [ ]:
from transformers import AutoTokenizer

LLAMA_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = ""

# توکنایزر LLaMA (instruction-tuned)
tokenizer_llama = AutoTokenizer.from_pretrained(
    LLAMA_ID,
    use_fast=False,
    token=HF_TOKEN
)

# برچسب‌های متنی که می‌خواهیم LLaMA تولید کند:
LABELS = {0:"neutral", 1:"entailment", 2:"contradiction"}

# تابعی برای ساخت پرامپت zero-shot
def build_zero_shot_prompt(premise, hypothesis):
    prompt = (
        "You are a helpful assistant that decides relationship between two sentences.\n"
        "Given a PREMISE and a HYPOTHESIS, answer with one word: 'entailment', 'contradiction' or 'neutral'.\n\n"
        f"PREMISE: {premise}\n"
        f"HYPOTHESIS: {hypothesis}\n"
        "Answer:"
    )
    return prompt

# تابع برای ساخت پرامپت one-shot
def build_one_shot_prompt(demo_premise, demo_hypothesis, demo_label, premise, hypothesis):
    demo = (
        f"PREMISE: {demo_premise}\n"
        f"HYPOTHESIS: {demo_hypothesis}\n"
        f"Answer: {demo_label}\n\n"
    )
    prompt = (
        "You are a helpful assistant that decides relationship between two sentences.\n"
        "Given a PREMISE and a HYPOTHESIS, answer with one word: 'entailment', 'contradiction' or 'neutral'.\n\n"
        + demo +
        f"PREMISE: {premise}\n"
        f"HYPOTHESIS: {hypothesis}\n"
        "Answer:"
    )
    return prompt



# ***Zero-shot evaluation (on a small eval_subset) — inference with generation***

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# شناسه مدل
LLAMA_GEN_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = ""

# بارگذاری توکنایزر با توکن
tokenizer_llama = AutoTokenizer.from_pretrained(
    LLAMA_GEN_ID,
    use_auth_token=HF_TOKEN
)

# بارگذاری مدل با توکن
model_llama = AutoModelForCausalLM.from_pretrained(
    LLAMA_GEN_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    use_auth_token=HF_TOKEN
)
model_llama.eval()


# تابع inference ساده با تنظیمات temperature و max_new_tokens
def llama_generate_text(prompt, temperature=0.0, max_new_tokens=16):
    inputs = tokenizer_llama(prompt, return_tensors="pt").to(model_llama.device)
    with torch.no_grad():
        out = model_llama.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.0
        )
    return tokenizer_llama.batch_decode(
        out[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0].strip()


# اجرای zero-shot روی یک زیرمجموعه کوچک و محاسبهٔ دقت
def run_zero_shot_eval(dataset, temperature=0.0, max_new_tokens=16):
    preds = []
    refs = []
    for ex in dataset:
        p = ex["premise"]
        h = ex["hypothesis"]
        prompt = build_zero_shot_prompt(p, h)
        text_out = llama_generate_text(prompt, temperature=temperature, max_new_tokens=max_new_tokens)
        # برداشت اولین کلمه خروجی و نرمال سازی
        pred_label = text_out.split()[0].lower().strip().strip(".")
        # map به برچسب‌های هدف
        mapped = None
        for k, v in LABELS.items():
            if v.startswith(pred_label):
                mapped = k
                break
        if mapped is None:
            # fallback: neutral
            mapped = 0
        preds.append(mapped)
        refs.append(ex["label"])
    return preds, refs


# -----------------------
# اینجا: کوچکتر کردن eval_subset
# -----------------------
EVAL_SIZE = 20

# سازگاری با انواع مختلف (HuggingFace Dataset یا لیست/iterator)
try:
    from datasets import Dataset as HFDataset
    is_hf_dataset = isinstance(eval_subset, HFDataset)
except Exception:
    is_hf_dataset = False

if is_hf_dataset:
    # اگر از اندازه بزرگتر باشه، اولین EVAL_SIZE نمونه رو انتخاب کن
    if len(eval_subset) > EVAL_SIZE:
        eval_subset_small = eval_subset.select(range(EVAL_SIZE))
    else:
        eval_subset_small = eval_subset
else:
    # اگر لیست-مانند باشه یا iterator
    try:
        # لیست یا sequence قابل indexing
        if len(eval_subset) > EVAL_SIZE:
            eval_subset_small = eval_subset[:EVAL_SIZE]
        else:
            eval_subset_small = eval_subset
    except Exception:
        # fallback امن برای iteratorها
        import itertools
        eval_subset_small = list(itertools.islice(eval_subset, EVAL_SIZE))

print(f"Using eval subset size: {len(eval_subset_small)}")


# اجرا با subset کوچک‌تر (بدون تغییر در منطق بعدی)
preds_zero, refs_zero = run_zero_shot_eval(eval_subset_small, temperature=0.0, max_new_tokens=8)
acc_zero = acc_metric.compute(predictions=preds_zero, references=refs_zero)
print("Zero-shot accuracy (temperature=0.0):", acc_zero)



# ***One-shot evaluation (select a demonstration and run on the same eval_subset)***

In [ ]:
# انتخاب نمونهٔ دمو از train10pct (یک نمونه)
demo = train10pct.shuffle(seed=999).select(range(1))[0]  # یک نمونه
demo_premise = demo["premise"]
demo_hypothesis = demo["hypothesis"]
demo_label_text = LABELS[demo["label"]]

print("Demo:", demo_premise, "|", demo_hypothesis, "->", demo_label_text)

# اجرای one-shot evaluation با زیرمجموعه کوچک
def run_one_shot_eval(dataset, demo, temperature=0.0, max_new_tokens=8, subset_size=20):
    # انتخاب زیرمجموعه کوچک
    try:
        from datasets import Dataset as HFDataset
        is_hf_dataset = isinstance(dataset, HFDataset)
    except Exception:
        is_hf_dataset = False

    if is_hf_dataset:
        eval_subset_small = dataset.select(range(min(len(dataset), subset_size)))
    else:
        try:
            eval_subset_small = dataset[:subset_size]
        except Exception:
            import itertools
            eval_subset_small = list(itertools.islice(dataset, subset_size))

    print(f"Using one-shot eval subset size: {len(eval_subset_small)}")

    preds = []
    refs = []
    dp = demo["premise"]
    dh = demo["hypothesis"]
    dl = LABELS[demo["label"]]

    for ex in eval_subset_small:
        p = ex["premise"]
        h = ex["hypothesis"]
        prompt = build_one_shot_prompt(dp, dh, dl, p, h)
        out = llama_generate_text(prompt, temperature=temperature, max_new_tokens=max_new_tokens)
        pred_label = out.split()[0].lower().strip().strip(".")
        mapped = None
        for k, v in LABELS.items():
            if v.startswith(pred_label):
                mapped = k
                break
        if mapped is None:
            mapped = 0  # fallback
        preds.append(mapped)
        refs.append(ex["label"])

    return preds, refs

# اجرا با زیرمجموعه کوچک‌تر
preds_one, refs_one = run_one_shot_eval(eval_subset, demo, temperature=0.0, max_new_tokens=8, subset_size=20)
acc_one = acc_metric.compute(predictions=preds_one, references=refs_one)
print("One-shot accuracy (temperature=0.0):", acc_one)


# ***QLoRA***

In [ ]:
# -------------------------------
# QLoRA Training (4-bit) + Eval روی MultiNLI
# -------------------------------

!pip install -q bitsandbytes transformers peft datasets accelerate evaluate

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    default_data_collator, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import evaluate

# -------------------------------
# HuggingFace model & token
# -------------------------------
LLAMA_GEN_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = ""
EVAL_SAMPLE = 20

# -------------------------------
# 4-bit Quantization Config
# -------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# -------------------------------
# Load model (QLoRA)
# -------------------------------
device_map = "auto" if torch.cuda.is_available() else {"": "cpu"}

model_q = AutoModelForCausalLM.from_pretrained(
    LLAMA_GEN_ID,
    device_map=device_map,
    quantization_config=bnb_config,
    use_auth_token=HF_TOKEN
)
model_q.eval()

# Tokenizer
tokenizer_llama = AutoTokenizer.from_pretrained(
    LLAMA_GEN_ID,
    use_auth_token=HF_TOKEN
)
tokenizer_llama.pad_token = tokenizer_llama.eos_token

# -------------------------------
# LoRA Config
# -------------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_q = get_peft_model(model_q, lora_config)
model_q.print_trainable_parameters()

# -------------------------------
# بارگذاری دیتاست MultiNLI
# -------------------------------
raw = load_dataset("multi_nli")

train_full = raw["train"]
val_matched = raw["validation_matched"]

train_small = train_full.shuffle(seed=42).select(range(int(len(train_full)*0.01)))  # فقط 1٪
eval_subset = val_matched.shuffle(seed=42).select(range(min(EVAL_SAMPLE, len(val_matched))))

print("Train size:", len(train_small))
print("Eval subset size:", len(eval_subset))

# -------------------------------
# Label mapping
# -------------------------------
LABELS = {0: "entailment", 1: "neutral", 2: "contradiction"}

# -------------------------------
# Tokenize function
# -------------------------------
def tokenize_for_causal(examples):
    prompts = [
        f"Decide the relation between sentences.\nPREMISE: {p}\nHYPOTHESIS: {h}\nAnswer:"
        for p, h in zip(examples["premise"], examples["hypothesis"])
    ]
    out = tokenizer_llama(prompts, truncation=True, max_length=128, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

train_tok = train_small.map(tokenize_for_causal, batched=True, remove_columns=train_small.column_names)
train_tok.set_format("torch")

# -------------------------------
# TrainingArguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./results/qlora_llama",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=1,
    save_strategy="no",
    push_to_hub=False,
    report_to="none",
)

# -------------------------------
# Trainer
# -------------------------------
trainer_q = Trainer(
    model=model_q,
    args=training_args,
    train_dataset=train_tok,
    tokenizer=tokenizer_llama,
    data_collator=default_data_collator,
)

trainer_q.train()

# -------------------------------
# Simple generation function
# -------------------------------
def llama_generate_text(prompt, temperature=0.0, max_new_tokens=16):
    inputs = tokenizer_llama(prompt, return_tensors="pt").to(model_q.device)
    with torch.no_grad():
        out = model_q.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.0
        )
    return tokenizer_llama.batch_decode(
        out[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0].strip()

# -------------------------------
# Eval function (اصلاح‌شده)
# -------------------------------
acc_metric = evaluate.load("accuracy")

def eval_causal_model_on_dataset(dataset, temperature=0.0):
    preds, refs = [], []
    for ex in dataset:
        prompt = f"Decide the relation between sentences.\nPREMISE: {ex['premise']}\nHYPOTHESIS: {ex['hypothesis']}\nAnswer:"
        out = llama_generate_text(prompt, temperature=temperature, max_new_tokens=8)

        # بررسی خالی بودن خروجی مدل
        if not out.strip():
            mapped = 1  # پیش‌فرض neutral
        else:
            pred_label = out.split()[0].lower().strip().strip(".")
            mapped = None
            for k, v in LABELS.items():
                if v.startswith(pred_label):
                    mapped = k
                    break
            if mapped is None:
                mapped = 1  # پیش‌فرض neutral

        preds.append(mapped)
        refs.append(ex["label"])
    return preds, refs

# -------------------------------
# Run eval
# -------------------------------
preds_q, refs_q = eval_causal_model_on_dataset(eval_subset, temperature=0.0)
acc = acc_metric.compute(predictions=preds_q, references=refs_q)
print("QLoRA eval accuracy:", acc)



# QLoRA + Linear layer

In [ ]:
# -------------------------------
# 1️⃣ نصب کتابخانه‌ها و نسخه‌های سازگار
# -------------------------------
!pip uninstall -y transformers bitsandbytes peft
!pip install transformers==4.33.1 peft==0.5.0 bitsandbytes==0.43.1 datasets evaluate accelerate

# -------------------------------
# 2️⃣ ایمپورت‌ها
# -------------------------------
import torch
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    default_data_collator, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import evaluate

# -------------------------------
# 3️⃣ تنظیمات مدل و HuggingFace token
# -------------------------------
LLAMA_GEN_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = ""
EVAL_SAMPLE = 20

# -------------------------------
# 4️⃣ 4-bit Quantization Config
# -------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# -------------------------------
# 5️⃣ بارگذاری مدل پایه و آماده‌سازی برای QLoRA
# -------------------------------
device_map = "auto" if torch.cuda.is_available() else {"": "cpu"}

base_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_GEN_ID,
    device_map=device_map,
    quantization_config=bnb_config,
    token=HF_TOKEN
)
base_model = prepare_model_for_kbit_training(base_model)

# -------------------------------
# 6️⃣ تعریف wrapper با لایه خطی برای Classification
# -------------------------------
class LlamaForNLI(nn.Module):
    def __init__(self, base_model, hidden_size, num_labels=3):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.base.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        # گرفتن hidden state آخرین توکن [CLS-like]
        pooled = outputs.last_hidden_state[:, -1, :]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

hidden_size = base_model.config.hidden_size
model_linear = LlamaForNLI(base_model, hidden_size=hidden_size, num_labels=3)

# -------------------------------
# 7️⃣ اعمال QLoRA روی لایه خطی
# -------------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["classifier"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
model_linear = get_peft_model(model_linear, lora_config)
model_linear.print_trainable_parameters()

# -------------------------------
# 8️⃣ توکنایزر
# -------------------------------
tokenizer_llama = AutoTokenizer.from_pretrained(
    LLAMA_GEN_ID,
    token=HF_TOKEN
)
tokenizer_llama.pad_token = tokenizer_llama.eos_token

# -------------------------------
# 9️⃣ بارگذاری دیتاست MultiNLI
# -------------------------------
raw = load_dataset("multi_nli")
train_full = raw["train"]
val_matched = raw["validation_matched"]

train_small = train_full.shuffle(seed=42).select(range(int(len(train_full)*0.01)))
eval_subset = val_matched.shuffle(seed=42).select(range(min(EVAL_SAMPLE, len(val_matched))))

print("Train size:", len(train_small))
print("Eval subset size:", len(eval_subset))

# -------------------------------
# 🔟 توکنایز کردن داده‌ها
# -------------------------------
def tokenize_for_classification(examples):
    prompts = [
        f"PREMISE: {p}\nHYPOTHESIS: {h}"
        for p, h in zip(examples["premise"], examples["hypothesis"])
    ]
    out = tokenizer_llama(
        prompts,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    out["labels"] = examples["label"]
    return out

train_tok = train_small.map(tokenize_for_classification, batched=True, remove_columns=train_small.column_names)
eval_tok = eval_subset.map(tokenize_for_classification, batched=True, remove_columns=eval_subset.column_names)

train_tok.set_format("torch")
eval_tok.set_format("torch")

# -------------------------------
# 1️⃣1️⃣ TrainingArguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./results/qlora_llama_linear",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    evaluation_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

# -------------------------------
# 1️⃣2️⃣ تعریف metric
# -------------------------------
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return acc_metric.compute(predictions=preds, references=labels)

# -------------------------------
# 1️⃣3️⃣ Trainer
# -------------------------------
trainer = Trainer(
    model=model_linear,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=tokenizer_llama,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics
)

# -------------------------------
# 1️⃣4️⃣ آموزش
# -------------------------------
trainer.train()

# -------------------------------
# 1️⃣5️⃣ ذخیره مدل
# -------------------------------
try:
    merged = model_linear.merge_and_unload()
    merged.save_pretrained("./results/qlora_llama_linear_merged")
    print("Merged and saved linear-QLoRA model.")
except Exception as e:
    print("Could not merge linear model; saved adapter only. Err:", e)
    model_linear.save_pretrained("./results/qlora_llama_linear_adapter")
